In [ ]:
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
from PIL import Image
import torch
from transformers import Sam3Processor, Sam3Model, pipeline
from huggingface_hub import login
import cv2

from google.colab import userdata

Setting up the transformation pipeline.

In [ ]:
def get_train_transforms_v2():
    """
    Returns transformations for the training set.
    Includes augmentations suitable for cell/cytology images (rotation invariant)
    and resizing to 1008x1008 for optimal SAM3 backbone alignment.
    """
    return A.Compose([
        # Resize to 1008x1008 to match SAM3 patch size (14 * 72 = 1008)
        # This avoids padding issues in the model.
        A.Resize(height=1008, width=1008),

        # Cytology images are usually rotation invariant
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),

        A.Affine(
          scale=(0.95, 1.05),      # Scale between 95% and 105%
          translate_percent=0.02,  # Shift up to 5%
          rotate=(-5, 5),        # Rotation between -15 and 15 degrees
          shear=(-1.2, 1.2),           # Very slight stretching (mimics slide tilt)
          p=0.2
        ),

        A.GaussNoise(std_range=(0.075, 0.12), p=0.4),

        A.CoarseDropout(
          num_holes_range=(1, 8),
          hole_height_range=(10, 28),
          hole_width_range=(10, 28),
          fill=0,
          p=0.1
        ),

        # RB and RC: Adjusts lighting and sharpness of your PAP smear images
        A.RandomBrightnessContrast(
            brightness_limit=0.12,
            contrast_limit=0.12,
            p=0.2
        ),

        # Normalize to 0-1 and convert to Tensor
        # Note: Mean/Std normalization is typically handled by the FasterRCNN model internal transform
        # checking against ImageNet stats. We output 0-1 float tensors here.
        A.ToFloat(max_value=255.0),
        ToTensorV2()
    ]
                     #, bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'])
    )

Loading model and testing SAM3's segmentation on a RIVA image. Testing how does the "cell" concept prompt work.

In [ ]:
# Authenticate with Hugging Face (Replace with your token)
login(token="")

# Load the SAM 3 processor and model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading SAM 3 on {device}...")

model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")

# Load image and define the text prompt
image_path = "./LSIL_6_2.png"
image = Image.open(image_path).convert("RGB")

transforms = get_train_transforms_v2()

# Apply transformations
# SAM 3 usually needs the original image as a numpy array for Albumentations
image_np = np.array(image)
transformed = transforms(image=image_np)
transformed_image = transformed["image"]

# Pass the transformed image to the processor
# Note: If your transform outputs a Tensor, convert it back to PIL
# or a clean NumPy array so the Sam3Processor can format it for the model.
if isinstance(transformed_image, torch.Tensor):
    # Convert CHW Tensor to HWC Numpy for the processor
    transformed_for_proc = transformed_image.permute(1, 2, 0).cpu().numpy()
else:
    transformed_for_proc = transformed_image

# Prepare inputs (The processor handles the text-image alignment)
inputs = processor(
    images=transformed_for_proc,
    text=text_prompt,
    return_tensors="pt"
).to(device)

# Run Inference
with torch.no_grad():
    outputs = model(**inputs)

# What do you want to segment? (SAM 3's open-vocabulary feature)
text_prompt = "cells"

# Prepare inputs and run inference
inputs = processor(images=image, text=text_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# Post-process the masks
# The output contains predicted masks and confidence scores
results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

print(f"Detected {len(results)} instances of '{text_prompt}'")

# Visualization (Overlaying masks on the original image)
image_cv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
color = np.array([0, 255, 0], dtype='uint8') # Green masks

for mask in results["masks"]:
    # Convert boolean mask to binary image
    mask_np = mask.cpu().numpy().squeeze()

    # Apply semi-transparent green overlay
    colored_mask = np.zeros_like(image_cv)
    colored_mask[mask_np > 0] = color
    image_cv = cv2.addWeighted(image_cv, 1, colored_mask, 0.5, 0)

# Save and display the result
cv2.imwrite("sam3_output2.jpg", image_cv)
print("Saved segmentation output to sam3_output.jpg")

Defining a ground truth box visualization function for a single test image.

In [ ]:
# Load the image
image_path = "./LSIL_6_2.png"
image = cv2.imread(image_path)

if image is None:
    raise FileNotFoundError(f"Could not find the image at {image_path}. Please check the path.")

# Define the raw data provided from the train.csv
raw_data = """LSIL_6_2.png,906.7480916030536,995.3384223918576,100,100,LSIL,4
LSIL_6_2.png,31.267175572519083,911.9592875318068,100,100,HSIL,5
LSIL_6_2.png,312.67175572519085,729.5674300254453,100,100,HSIL,5
LSIL_6_2.png,317.882951653944,846.8193384223919,100,100,HSIL,5
LSIL_6_2.png,190.2086513994912,776.4681933842239,100,100,NILM,0
LSIL_6_2.png,333.5165394402036,898.9312977099237,100,100,HSIL,5
LSIL_6_2.png,302.24936386768445,935.409669211196,100,100,HSIL,5
LSIL_6_2.png,257.9541984732825,914.5648854961833,100,100,HSIL,5
LSIL_6_2.png,78.16793893129771,562.8091603053437,100,100,HSIL,5
LSIL_6_2.png,211.05343511450383,469.00763358778624,100,100,HSIL,5
LSIL_6_2.png,857.2417302798982,502.88040712468194,100,100,HSIL,5
LSIL_6_2.png,479.43002544529264,403.86768447837153,100,100,LSIL,4"""

# Define a color dictionary (BGR format for OpenCV)
class_colors = {
    "NILM": (0, 255, 0),    # Green
    "LSIL": (0, 165, 255),  # Orange
    "HSIL": (0, 0, 255)     # Red
}

# Process each line of the data
lines = raw_data.strip().split('\n')

for line in lines:
    parts = line.split(',')

    # Extract coordinates and dimensions (converting floats to integers for pixel indexing)
    x = int(float(parts[1]))
    y = int(float(parts[2]))
    w = int(float(parts[3]))
    h = int(float(parts[4]))

    # Extract class info
    class_name = parts[5]

    # Get the color for this class (default to white if not found)
    color = class_colors.get(class_name, (255, 255, 255))

    # Calculate bottom-right corner
    x2 = x + w
    y2 = y + h

    # Draw the bounding box
    cv2.rectangle(image, (x, y), (x2, y2), color, thickness=2)

    # Add the class label above the box
    # Create a filled rectangle as background for the text to make it readable
    text = class_name
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.6
    font_thickness = 2

    # Get text size
    (text_w, text_h), baseline = cv2.getTextSize(text, font, font_scale, font_thickness)

    # Draw text background
    cv2.rectangle(image, (x, y - text_h - 5), (x + text_w, y), color, -1)

    # Put text (in white or black depending on color contrast)
    text_color = (255, 255, 255) if class_name == "HSIL" else (0, 0, 0)
    cv2.putText(image, text, (x, y - 5), font, font_scale, text_color, font_thickness)

# Save and display the result
output_filename = "LSIL_6_2_visualized.png"
cv2.imwrite(output_filename, image)
print(f"Image saved as {output_filename}")